### Build the AnnData object and filter peaks
Input: one `.h5ad` file per sample from `pipeline/04_annotation/053_isletepigenetics_export_to_anndata.R` and `data/public/scATACseq/Project/meta.csv` (`054`).
Output: `full_adata_filtered_10.h5ad` (peaks detected in more than 10% of the cells of at least one cell type) and `IC_metadata.csv`, read by `PeakANVI_donor_model.ipynb`.

In [ ]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import os

### Concatenate the per-sample objects

In [ ]:
# Get a list of all files in the directory
files = os.listdir('../../data/public/scATACseq/Project/AnnData/')
files = [item for item in files if 'h5ad' in item]
os.makedirs('../../data/public/scATACseq/Project/CorrectedObjects/', exist_ok = True)

# Load each data object, keep the dimension reductions and save without .raw or .obsm
for idx in range(len(files)):
    # Read the data
    data = ad.read_h5ad('../../data/public/scATACseq/Project/AnnData/' + files[idx])

    # Extract dimension reduction objects
    if idx == 0:
        harmony = data.obsm['X_Harmony']
        LSI = data.obsm['X_LSI']
        UMAP = data.obsm['X_UMAP']
        cellnames = np.array(data.obs.index.tolist())
    else:
        harmony = np.concatenate((harmony, data.obsm['X_Harmony']))
        LSI = np.concatenate((LSI, data.obsm['X_LSI']))
        UMAP = np.concatenate((UMAP, data.obsm['X_UMAP']))
        cellnames = np.concatenate((cellnames, np.array(data.obs.index.tolist())))

    # Write an AnnData object without the .raw or .obsm
    data.raw = None
    data.obsm = None
    data.write('../../data/public/scATACseq/Project/CorrectedObjects/' + files[idx])

In [ ]:
# Get a list of all files in the directory
files = os.listdir('../../data/public/scATACseq/Project/CorrectedObjects/')
files = [item for item in files if 'h5ad' in item]
files = ['../../data/public/scATACseq/Project/CorrectedObjects/' + item for item in files]
# Subset list of files because concat_on_disk on all files leads to loss of fragments
files_a = files[0:61]
files_b = files[61:122]

In [ ]:
# Concatenate on disk in batches,
# Concatenate all on disk results in missing fragments
ad.experimental.concat_on_disk(in_files = files_a, out_file = '../../data/public/scATACseq/Project/a_adata.h5ad')
ad.experimental.concat_on_disk(in_files = files_b, out_file = '../../data/public/scATACseq/Project/b_adata.h5ad')

In [ ]:
# Concatenate in memory and save on disk
ann_a = sc.read_h5ad(filename = '../../data/public/scATACseq/Project/a_adata.h5ad')
ann_b = sc.read_h5ad(filename = '../../data/public/scATACseq/Project/b_adata.h5ad')
ann_c = ad.concat([ann_a, ann_b])
ann_c.write('../../data/public/scATACseq/Project/full_adata.h5ad')

### Add dimension reductions and donor metadata

In [ ]:
## Add dimensional reduction information
# Read the final dataset (file backed)
ann = ad.read_h5ad('../../data/public/scATACseq/Project/full_adata.h5ad', backed = True)

# Get list of cells
cells = np.array(ann.obs.index.tolist())

# Get sort order
index_dict = {value: index for index, value in enumerate(cellnames)}
indices = np.array([index_dict[value] for value in cells])

# Insert dimensional reductions
ann.obsm['X_Harmony'] = harmony[indices,:]
ann.obsm['X_LSI'] = LSI[indices,:]
ann.obsm['X_umap'] = UMAP[indices,:]

# Write object
ann.write('../../data/public/scATACseq/Project/full_adata_dr.h5ad')

In [ ]:
ann = ad.read_h5ad('../../data/public/scATACseq/Project/full_adata_dr.h5ad', backed = True)
meta = pd.read_csv('../../data/public/scATACseq/Project/meta.csv')
ann.obs = ann.obs.merge(meta[['Sample', 'disease', 'gender', 'age', 'ethnicity', 'bmi', 'hba1c']], on='Sample', how='left')

# Write object
ann.write('../../data/public/scATACseq/Project/full_adata_dr_meta.h5ad')

### Filter peaks

In [ ]:
adata = sc.read_h5ad(filename = '../../data/public/scATACseq/Project/full_adata_dr_meta.h5ad')
adata

In [ ]:
# Find cell types
cell_types = adata.obs.labels.unique()

# Create empty dataframe of each peak in each cell type
prop = pd.DataFrame(columns = cell_types, index = adata.var_names)

# Fill data frame with proportions of cells containing feagment by cell type
for c in cell_types:
    s = adata.X[adata.obs.labels == c]
    prop[c] = np.array((s > 0).sum(axis = 0)/ s.shape[0]).flatten()

prop

In [ ]:
# Create index of which peaks to keep (detected in more than 10% of the cells of any cell type)
idx_keep = prop.apply(lambda row: any(row > 0.10), axis = 1)

# Keep only peaks in index
adata = adata[:,idx_keep]

# Save anndata object to new file
adata.write('../../data/public/scATACseq/Project/full_adata_filtered_10.h5ad')

### Cell metadata

In [ ]:
# Obtain metadata
adata.obs.to_csv('../../data/public/scATACseq/Project/IC_metadata.csv')